# Resampling Methods

Astro 128 (UC Berkeley, 2026)


## A Brief History of Resampling Methods

The two resampling techniques we'll use today — the **jackknife** and the **bootstrap** — were born decades apart but are closely related. Both address the same fundamental question: *how can we estimate the uncertainty of a statistic without strong assumptions about the underlying distribution?*

### The Jackknife (1949–1958)

The jackknife was introduced by **Maurice Quenouille** in 1949 as a method for reducing bias in estimators. It was later extended and named by **John Tukey** in 1958, who called it the "jackknife" because, like a Boy Scout's jackknife, it's a rough-and-ready general-purpose tool. Tukey showed that the leave-one-out procedure could also be used to estimate variance, not just correct bias.

### The Bootstrap (1979)

The bootstrap was invented by **Bradley Efron** (Stanford) in his landmark 1979 paper, [*"Bootstrap Methods: Another Look at the Jackknife."*](https://projecteuclid.org/journals/annals-of-statistics/volume-7/issue-1/Bootstrap-Methods-Another-Look-at-the-Jackknife/10.1214/aos/1176344552.full). The name comes from the phrase "pulling yourself up by your own bootstraps": the idea that you can learn about the sampling distribution *from the sample itself*, without knowing the true population. Efron showed that resampling with replacement was both more flexible and more powerful than the jackknife for most problems, essentially generalizing Tukey's idea.

### The Berkeley Connection

Resampling methods have deep roots at Berkeley. **Peter Bickel** (UC Berkeley Statistics) did foundational theoretical work establishing *when* and *why* the bootstrap works and, importantly, when it fails, leading to corrected methods like the "$m$ out of $n$" bootstrap. **David Freedman** (UC Berkeley Statistics) provided rigorous analysis of the bootstrap's consistency properties and was an influential advocate for nonparametric and resampling-based inference over parametric assumptions. Their work helped transform the bootstrap from a clever idea into a theoretically grounded tool that is now ubiquitous in science.

## Setup: Synthetic Linear Data

We start with a simple test case: a line $y = mx + b$ with known parameters, observed at a handful of points with Gaussian noise. With only 6 data points, the effects of resampling will be easy to visualize.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Make some (linear) data
NDATA = 6
SIG_NOISE = 0.3
XRANGE = (-1, 1)

def func(x, m=2.2, b=-0.7):
    return m*x + b
    
rng = np.random.default_rng(0)
x = rng.uniform(XRANGE[0], XRANGE[1], size=NDATA)
n = rng.normal(size=NDATA, scale=SIG_NOISE)
y_true = func(x)
y_meas = y_true + n

### Initial fit

First we do a standard polynomial fit to the full dataset. This gives us a single best-fit line — but no error bars on the parameters. The question driving this notebook is: **how uncertain are `m` and `b`?**

In [ ]:
# Fit a line
DEG = 1
poly = np.polyfit(x, y_meas, deg=DEG)

In [ ]:
plt.figure()
plt.plot(x, y_meas, '.')
plt.plot(XRANGE, func(np.array(XRANGE)), ':', label='true')
plt.plot(XRANGE, np.polyval(poly, np.array(XRANGE)), 'k-', label='fit')
plt.xlim(*XRANGE)
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid()

## Bootstrap Resampling

The **bootstrap** is a powerful resampling technique for estimating the uncertainty of a statistic (e.g., fit parameters) when analytic error bars are hard to derive.

The idea is simple: given $N$ data points, generate many "bootstrap samples" by drawing $N$ points **with replacement** from the original data. Fit each bootstrap sample, and the spread of the resulting parameters gives you an estimate of the uncertainty.

Key properties:
- Each bootstrap sample has the same size as the original data, but some points appear multiple times and some not at all.
- The method is **nonparametric**: it makes no assumptions about the underlying distribution of the data.
- Works well when $N$ is large enough to be representative of the true population.

See also [`scipy.stats.bootstrap`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html) for a production-ready implementation.

In [ ]:
# Example 1: bootstrap resampling
NBOOTS = 1000
rng = np.random.default_rng(1)

polys_boot = []
for i in range(NBOOTS):
    inds_boot = rng.integers(NDATA, size=NDATA)
    poly = np.polyfit(x[inds_boot], y_meas[inds_boot], deg=DEG)
    polys_boot.append(poly)

### Summarizing the bootstrap distribution

Each bootstrap iteration produced a different `(m, b)`. We summarize the spread by computing the mean and the 5th/95th percentiles (a 90% confidence interval) across all 1000 bootstrap fits.

In [ ]:
polys_boot = np.array(polys_boot)
polys_boot_mean = np.mean(polys_boot, axis=0)  # ONLY WORKS IF LINEAR
polys_boot_lo = np.percentile(polys_boot, 5, axis=0)
polys_boot_hi = np.percentile(polys_boot, 95, axis=0)
for i in range(DEG + 1):
    print(f"{polys_boot_mean[i]:+5.3f} [{polys_boot_lo[i]:+5.3f}, {polys_boot_hi[i]:+5.3f}]")

### Visualizing bootstrap uncertainty

Overplotting all 1000 bootstrap fit lines shows the "envelope" of plausible fits. The spread of lines is widest near the edges of the data range, as you'd expect — extrapolation amplifies parameter uncertainty.

In [ ]:
plt.figure()
plt.plot(x, y_meas, '.')
plt.plot(XRANGE, func(np.array(XRANGE)), ':')
for poly in polys_boot:
    plt.plot(XRANGE, np.polyval(poly, np.array(XRANGE)), 'k-', alpha=2/NBOOTS)
plt.xlim(*XRANGE)
plt.ylim(-5, 5)
plt.xlabel('x')
plt.ylabel('y')
plt.grid()

## Jackknife Resampling

The **jackknife** (also called leave-one-out resampling) is an older resampling technique that predates the bootstrap. For $N$ data points, you create $N$ resampled datasets, each formed by dropping one of the original points.

Compared to the bootstrap:
- The jackknife is **deterministic** (no random sampling involved).
- It produces fewer resampled datasets ($N$ vs. as many as you want from bootstrapping).
- It can also be used to identify **influential data points**: if removing a single point changes the result dramatically, that point has high leverage.

In [ ]:
# Example 2: Jackknife resampling (leave-one-out)
polys_jack = []
inds = np.arange(NDATA)
for i in range(NDATA):
    poly = np.polyfit(x[inds != i], y_meas[inds != i], deg=DEG)
    polys_jack.append(poly)

In [ ]:
plt.figure()
plt.plot(x, y_meas, '.')
plt.plot(XRANGE, func(np.array(XRANGE)), ':')
for poly in polys_jack:
    plt.plot(XRANGE, np.polyval(poly, np.array(XRANGE)), 'k-', alpha=2/NDATA)
plt.xlim(*XRANGE)
plt.xlabel('x')
plt.ylabel('y')
plt.grid()

### Identifying influential points

A key advantage of the jackknife: we can ask *which data point matters most?* Below, we compare each point's jackknife residual (how much the fit changes when that point is dropped) to its actual noise. Points where the jackknife deviation is much larger than the noise have high **leverage** — they disproportionately control the fit.

In [ ]:
fig, axes = plt.subplots(nrows=2, sharex=True)
err = [y_meas[i] - np.polyval(poly, x[i]) for i, poly in enumerate(polys_jack)]
axes[0].plot(x, y_meas, '.')
axes[1].plot(x, np.abs(err), '.', label='jackknife')
axes[1].plot(x, np.abs(y_meas - y_true), '.', label='noise')
axes[1].legend(loc='best')
axes[1].set_ylabel('Abs Deviation')
axes[1].set_xlabel('x')
for poly in polys_jack:
    axes[0].plot(XRANGE, np.polyval(poly, np.array(XRANGE)), 'k-', alpha=2/NDATA)
for ax in axes:
    ax.set_xlim(*XRANGE)
    ax.grid()

## A Warning About Weighting by Residuals

It is tempting to do "iterative reweighting" — fit a model, compute residuals, then reweight the data by $1/|{\rm residual}|^2$ and refit. **This is circular reasoning.** Points that happened to land close to the model (possibly by chance) get enormous weight, while points that may be perfectly valid measurements get downweighted just because the first fit wasn't great.

The result is a fit that is overly influenced by a few points and can be wildly wrong. If you want to do weighted fitting, the weights should come from **independent knowledge of the measurement uncertainties**, not from the residuals of the fit itself.

In [ ]:
# A warning about weighting

_poly = np.polyfit(x, y_meas, deg=DEG)
y_mdl = np.polyval(_poly, x)
wgt = 1 / np.abs(y_meas - y_mdl)**2
poly = np.polyfit(x, y_meas, w=wgt, deg=DEG)

plt.figure()
plt.plot(x, y_meas, '.')
plt.plot(XRANGE, func(np.array(XRANGE)), ':', label='true')
plt.plot(XRANGE, np.polyval(poly, np.array(XRANGE)), 'k-', label='weighted fit')
plt.xlim(*XRANGE)
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid()